# Прогноз потребления в МО и раннее обнаружение шоков

В этом ноутбуке мы проходим по решению шаг за шагом: от данных до найденных шоков. Задача — прогнозировать безналичные траты жителя в 2 190 муниципальных образованиях на 1, 3, 6 и 12 месяцев, обогнать Prophet и как можно раньше замечать структурные сдвиги.

Ноутбук запускается сверху вниз (**Run All**) и сам пересчитывает почти всё: панель данных, простые модели, Prophet, LightGBM, ансамбль, тесты значимости, детекторы шоков и графики. Весь прогон занимает около 20 минут на обычном ноутбуке.

Не пересчитываются только самые тяжёлые шаги — они идут часами:
- **дообучение нейросетей** перед каждой из 12 точек прогноза. Мы выложили веса Chronos-2 и TimesFM, дообученные на одной точке (конец 2023 года), — по ним ноутбук сам считает прогноз на весь 2024 год. Чтобы не захламлять репозиторий, остальные 11 копий весов мы не выкладываем: их получают дообучением заново (команда в README). Прогнозы нейросетей по всем точкам лежат готовыми в `results/forecasts/`;
- **сбор и разметка новостей** (сотни тысяч заголовков, языковая модель) — итоги в `results/news/`.

**Перед запуском:** `python -m venv .venv && .venv/bin/pip install -r requirements.txt`, затем откройте ноутбук с ядром из `.venv`.

**План:** 1. Данные · 2. Главная находка · 3. Как проверяем · 4. Модели · 5. Нейросети на коротких рядах · 6. Шоки · 7. Новости · 8. Ключевая ставка · 9. Итоги

In [ ]:
import os, sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, yaml, matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
FC = ROOT / "data/interim/forecasts"

def run(*args):
    """Запуск шага решения отдельным процессом (так же, как из терминала)."""
    import subprocess, time
    t = time.time()
    r = subprocess.run([sys.executable, "-W", "ignore", "-m", *args], capture_output=True, text=True)
    print(f"python -m {' '.join(args)}  — {time.time() - t:.0f} с")
    if r.returncode:
        print(r.stdout[-2000:], r.stderr[-3000:]); raise RuntimeError("шаг завершился с ошибкой")

from src.prepare import prepare
prepare()   # панель МО + готовые результаты тяжёлых шагов из results/

## 1. Данные
Главный файл — набор организаторов `consumption.parquet`. На каждый муниципалитет, месяц и категорию — одно число: сколько в среднем потратил по картам один житель.

In [ ]:
from src.data.load import build_panel
panel, terr = build_panel()
print(f"Строк: {len(panel):,}; МО: {panel.territory_id.nunique():,}; месяцев: {panel.period.nunique()} ({panel.period.min():%Y-%m} … {panel.period.max():%Y-%m})")
print("Категории:", ", ".join(panel.category.unique()))
print("МО с полной историей (24 мес.):", int((terr.months == 24).sum()))
panel[panel.category == "Все категории"].value.describe().round(0).to_frame("руб. на жителя в месяц").T

In [ ]:
from src.eval.cv import load_wide
Y = load_wide(yaml.safe_load(open("configs/eval.yaml")))
ref = pd.read_parquet("data/external/territory_reference.parquet").set_index("territory_id")
med = Y.median(axis=1)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(Y.index, med, lw=2, color="#2a78d6"); ax[0].set_title("Медианные траты жителя МО, руб./мес.")
mm = (Y.pct_change().median(axis=1) * 100).iloc[1:]
ax[1].bar(mm.index, mm.values, width=20, color=np.where(mm.values > 0, "#1baf7a", "#eb6834")); ax[1].set_title("Медианное изменение за месяц, % (сезонность)")
plt.tight_layout(); plt.show()

## 2. Главная находка: траты МО движутся вместе со страной
На графиках выше видно, что почти все муниципалитеты живут в одном ритме: в декабре траты взлетают, в январе проваливаются, а год к году растут примерно на 15 %. Своя динамика у МО — это проценты поверх общей.

Отсюда простая и сильная модель — **сезонная наивная модель с дрейфом**: берём тот же месяц год назад и умножаем на рост трат по России за последний известный год. Рост берём именно по стране: у муниципалитета 12–23 месяца истории, и свой годовой рост по ним не посчитать, а ряд России идёт с 2018 года.

In [ ]:
from src.forecast.lgbm_model import load_national
L = load_national(yaml.safe_load(open("configs/forecast.yaml")))
g = np.exp(L["2023-12-01"] - L["2022-12-01"])
mo = 1  # Майкоп
pred, fact = Y.loc["2023-12-01", mo] * g, Y.loc["2024-12-01", mo]
display(pd.DataFrame({"значение": [f"{Y.loc['2023-12-01', mo]:,.0f}", f"×{g:.3f}", f"{pred:,.0f}", f"{fact:,.0f}", f"{(pred/fact-1)*100:+.1f} %"]},
                     index=["Майкоп, декабрь 2023", "рост трат России за год", "прогноз на декабрь 2024", "факт", "ошибка"]))
fig, ax = plt.subplots(); ax.plot(np.exp(L), color="#0b0b0b"); ax.set_title("Траты по России, млрд руб./мес. — длинная история для роста и сезонности"); plt.show()

## 3. Как проверяем: скользящее окно
Модель в каждой точке прогноза видит только прошлое и прогнозирует 1, 3, 6 и 12 месяцев вперёд, потом мы сдвигаемся на месяц и повторяем. Ниже — то же самое вживую для двух простых моделей, а затем тест, который проверяет, что будущее в прогноз не попадает.

In [ ]:
from src.eval.cv import run_cv
from src.eval.metrics import summarize
from src.forecast.baselines import SeasonalNaive, SeasonalNaiveNationalGrowth
cfg = yaml.safe_load(open("configs/eval.yaml"))
res = pd.concat([run_cv(Y, SeasonalNaive(), cfg), run_cv(Y, SeasonalNaiveNationalGrowth(), cfg)])
print("Точек прогноза по горизонтам:", res[res.model == "seasonal_naive"].groupby("h").origin.nunique().to_dict())
summarize(res).pivot(index="model", columns="h", values="MAE").round(0)

In [ ]:
import subprocess
print(subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True).stdout[-200:])

## 4. Модели прогноза
Теперь считаем все модели на скользящем окне — каждая в отдельном процессе, на одних и тех же точках прогноза и муниципалитетах. Здесь пересчитываются сезонные наивные модели, Prophet и LightGBM (минут 10). Прогнозы нейросетей берутся из `results/forecasts/`, а в разделе 5 мы проверим их на выложенных весах.

In [ ]:
run("src.eval.run", "--models", "seasonal_naive", "snaive_natg", "snaive_shrink_w25_k1", "snaive_shrink_w50_k1",
    "snaive_shrink_w75_k1", "prophet", "lgbm_catmo", "lgbm_no_nat", "lgbm_resid")
run("src.eval.ensemble")       # ансамбль: веса моделей по ошибкам только на прошлых точках прогноза
run("src.eval.significance")   # тест Диболда–Мариано

Главная метрика — **MAE**, рублей на жителя в месяц. Дополнительно — R² по росту за год: угадывает ли модель, вырастут ли траты МО быстрее или медленнее среднего.

In [ ]:
NAMES = {"prophet": "Prophet (базовая)", "seasonal_naive": "Сезонная наивная", "snaive_natg": "Сезонная наивная с дрейфом",
         "snaive_shrink_w25_k1": "+ shrinkage роста МО (w=0,25)*", "autoets": "AutoETS", "autotheta": "AutoTheta", "nhits": "N-HiTS",
         "lgbm_catmo": "LightGBM", "lgbm_resid": "+ LightGBM по остаткам", "chronos2_bc": "Chronos-2 zero-shot + backcasting",
         "timesfm_bc": "TimesFM zero-shot + backcasting", "tirex_bc_mix": "TiRex + форма года МО", "timesfm_lora": "TimesFM дообученный (LoRA)",
         "chronos2_ft_mix": "Chronos-2 дообученный + форма года МО", "ens_inv_mae": "Ансамбль (веса по обратной ошибке)"}
have = [m for m in NAMES if (FC / f"{m}.parquet").exists()]
allres = pd.concat(pd.read_parquet(FC / f"{m}.parquet") for m in have)
S = summarize(allres)
tab = S.pivot(index="model", columns="h", values="MAE").loc[have].rename(index=NAMES).round(0).astype(int)
tab.columns = [f"{h} мес." for h in tab.columns]
display(tab.style.highlight_min(axis=0, color="#d6e8fb"))
print("* вес w подобран на тех же данных — оценка оптимистичная")

In [ ]:
from src.report import figures
figures.OUT.mkdir(parents=True, exist_ok=True); plt.rcParams["font.family"] = "DejaVu Sans"
figures.mae_by_horizon(); Image("report/figures/01_mae_by_horizon.png")

In [ ]:
r2 = S[S.model.isin(["prophet", "snaive_natg", "chronos2_ft_mix", "ens_inv_mae"])].pivot(index="model", columns="h", values="R2_yoy").rename(index=NAMES).round(2)
r2.columns = [f"R² роста, {h} мес." for h in r2.columns]; r2

In [ ]:
figures.example_forecast(); Image("report/figures/02_example_forecast.png")

**Значимы ли различия?** Тест Диболда–Мариано по точкам прогноза и доля муниципалитетов, где одна модель лучше другой. Точек немного (7–12), поэтому тест осторожный.

In [ ]:
sig = pd.read_csv("data/interim/significance.csv")
sig["A"] = sig.A.map(NAMES).fillna(sig.A); sig["B"] = sig.B.map(NAMES).fillna(sig.B)
sig[sig.h < 12][["A", "B", "h", "разница_MAE", "p_DM", "доля_МО_где_A_лучше"]]

**Какую модель выбрать.** Сравниваем по нескольким критериям сразу: точность на каждом горизонте, стабильность, время, понятность. Смотрим, кто недоминируем (Парето), и агрегируем точность правилами Борда и Коупленда из теории выбора.

In [ ]:
from src.eval import selection
selection.models()

Оба правила ставят первым ансамбль. Но на 3–12 месяцев он статистически не отличается от сезонной наивной модели с дрейфом, а та считается за доли секунды и объясняется одной фразой, — её мы и рекомендуем для практики. Дообученный Chronos-2 — лучшая нейросеть; интересно, где именно он полезен:

In [ ]:
rows = []
for m in ["snaive_natg", "chronos2_ft_mix", "ens_inv_mae"]:
    f = pd.read_parquet(FC / f"{m}.parquet"); f["ae"] = (f.y_true - f.y_pred).abs()
    f["история"] = np.where(((f.origin.dt.year - 2023) * 12 + f.origin.dt.month) <= 17, "12–17 мес.", "18–23 мес.")
    rows.append(f[f.h == 1].groupby("история").ae.mean().rename(NAMES[m]))
pd.concat(rows, axis=1).round(0).astype(int).T.rename_axis("MAE на 1 мес. при истории")

Chronos-2 выигрывает в первой половине 2024 года — тогда рост трат страны резко замедлился, и правило «рост сохранится» ошиблось. Когда всё успокоилось, простая модель его догнала. Поэтому в ансамбле нейросеть — страховка на случай смены режима.

## 5. Нейросети на коротких рядах
Фундаментальным моделям 12–23 точек мало (TimesFM официально нужно не меньше 32). Мы восстановили прошлое каждого МО по ряду России (**backcasting**), учли собственную **форму года** муниципалитета и **дообучили** модели перед каждой точкой прогноза. Пример — Соболевский район Камчатки, где траты поднимаются в октябре, после лососёвой путины:

In [ ]:
from src.forecast.foundation import _extend, _nat_level
nat = _nat_level(); tid = ref.index[ref.mo_name.str.contains("Соболевский")][0]
fig, ax = plt.subplots(figsize=(12, 4))
for mode, c in [("national", "#8b8a85"), ("mix", "#2a78d6")]:
    e = _extend(Y.iloc[:12][[tid]], nat, mode)
    ax.plot(e.index[-36:], e[tid].values[-36:], lw=2, color=c, label={"national": "backcasting: сезонность страны", "mix": "backcasting: смесь с формой года МО"}[mode])
ax.axvline(pd.Timestamp("2023-01-01"), color="k", ls="--", lw=1); ax.legend(); ax.set_title("Соболевский р-н Камчатки: восстановленное прошлое (до пунктира) и реальные данные 2023")
plt.show()

**Проверим на весах.** Ниже — Chronos-2 и TimesFM, дообученные на данных 2023 года. Ноутбук скачивает веса (около 240 МБ, релиз на GitHub) и сам считает прогноз на весь 2024 год. Веса выложены только для этой точки прогноза: для остальных 11 точек модели дообучаются заново (`python -m src.eval.run --models chronos2_ft_mix timesfm_lora`, несколько часов на CPU). Для TimesFM при первом запуске скачается и базовая модель с Hugging Face (около 900 МБ) — если не хочется ждать, поставьте `RUN_TIMESFM = False`.

Прогноз TimesFM по выложенному адаптеру совпадает с прогнозом из оценки до знака. У Chronos-2 он отличается в среднем на 1 %: дообучение на CPU не полностью детерминировано, а веса мы храним в половинной точности, чтобы архив весил вдвое меньше.

In [ ]:
from src.forecast import finetuned_weights as fw
RUN_TIMESFM = True
fw.fetch()
P = {"Chronos-2, дообученный": fw.forecast(fw.chronos())}
if RUN_TIMESFM:
    P["TimesFM, дообученный (LoRA)"] = fw.forecast(fw.timesfm())
origin = pd.Timestamp(fw.CFG["origin"])
for m, name in [("prophet", "Prophet"), ("snaive_natg", "Сезонная наивная с дрейфом")]:
    f = pd.read_parquet(FC / f"{m}.parquet")
    P[name] = f[f.origin == origin].pivot(index="target", columns="territory_id", values="y_pred")
fact = Y.loc[P["Prophet"].index]
mae = pd.DataFrame({k: (v.reindex(fact.index)[fact.columns] - fact).abs().mean(axis=1) for k, v in P.items()})
mae.index = [f"{i} мес." for i in range(1, len(mae) + 1)]
display(mae.T.round(0).astype(int).assign(среднее=lambda d: d.mean(axis=1).round(0).astype(int)))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for a, tid in zip(ax, [1, ref.index[ref.mo_name.str.contains("Соболевский")][0]]):
    a.plot(Y.index, Y[tid], color="#0b0b0b", lw=2, label="факт")
    for (k, v), c in zip(P.items(), ["#2a78d6", "#1baf7a", "#eb6834", "#8b8a85"]):
        a.plot(v.index, v[tid], color=c, lw=1.6, label=k)
    a.axvline(origin, color="k", ls="--", lw=1); a.set_title(f"{ref.mo_name.get(tid, tid)[:45]}: прогноз на 2024 год из точки «декабрь 2023»")
ax[0].legend(fontsize=8); plt.tight_layout(); plt.show()

Хотите увидеть само дообучение? Поставьте `FINETUNE = True`: Chronos-2 дообучится на рядах 2023 года (около 3 минут на CPU) и выдаст тот же прогноз.

In [ ]:
FINETUNE = False
if FINETUNE:
    from src.forecast.foundation import Chronos2FineTuned
    p = fw.forecast(Chronos2FineTuned(bc_mode="mix", name="chronos2_ft_mix"))
    print("MAE по месяцам 2024:", (p - fact).abs().mean(axis=1).round(0).astype(int).tolist())

Сводка по всем фундаментальным моделям на всех точках прогноза:

In [ ]:
fm = ["timesfm", "timesfm_bc", "timesfm_bc_mix", "timesfm_lora", "chronos2", "chronos2_bc", "chronos2_ft", "chronos2_ft_mix", "tirex_bc", "tirex_bc_mix", "prophet"]
fm = [m for m in fm if (FC / f"{m}.parquet").exists()]
t = summarize(pd.concat(pd.read_parquet(FC / f"{m}.parquet") for m in fm)).pivot(index="model", columns="h", values="MAE").loc[fm].round(0).astype(int)
t.columns = [f"{h} мес." for h in t.columns]; t

## 6. Обнаружение шоков
Мы сравнили шесть детекторов, все решают только по прошлому. Готовой разметки шоков нет, поэтому проверяли тремя способами: встраивали шоки известной величины в 400 реальных рядов МО (с одинаковой для всех долей ложных тревог — 10 %), смотрели на реальные кризисы России 2020 и 2022 годов и на реальные события в муниципалитетах 2024 года. Всё это считается здесь же:

In [ ]:
run("src.changepoint.benchmark")        # искусственные шоки
run("src.changepoint.national_events")  # кризисы России
run("src.changepoint.panel_shocks")     # шоки в МО за 2024 год

Выбираем детектор так же, как модель: недоминируемые по Парето и правила Борда и Коупленда по четырём критериям — пропуск шоков, задержка, тревоги до шока и реакция на разовый выброс.

In [ ]:
selection.detectors()

In [ ]:
bench = pd.read_csv("data/interim/changepoint/benchmark_summary.csv", index_col=0)
bench.sort_values("доля_найденных", ascending=False).round(3)

In [ ]:
figures.detectors(); Image("report/figures/03_detectors.png")

In [ ]:
figures.national_events(); Image("report/figures/05_national_events.png")

In [ ]:
sh = pd.read_parquet("data/interim/changepoint/panel_shocks.parquet")
best = sh[(sh.detector == "base_residual") & ~sh.seasonal & (sh.shift_pct.abs() >= 10)].copy()
best["МО"] = best.mo_name.str.replace("внутригородская территория города федерального значения", "").str[:40]
print(f"Шоков 2024 г. (лучший детектор, сдвиг ≥10 %, не сезонность): {len(best)} МО из {Y.shape[1]}")
best.sort_values("shift_pct", key=abs, ascending=False)[["МО", "region", "shift_month", "alarm", "shift_pct"]].head(10).round(1)

In [ ]:
figures.shock_examples(); Image("report/figures/04_shock_examples.png")

## 7. Новости
Новость — это момент и место в тексте, а данные СберИндекса — месяц и код муниципалитета. Согласуем их в три шага: **время** (новость относится к месяцу публикации, прогноз видит только новости до своего момента), **место** (город в заголовке → МО и регион, с учётом падежей), **частота** (поток новостей → месячные доли тем). Источники — Lenta.ru, ИА REGNUM и МЧС России.

Сами заголовки — чужие тексты, поэтому в репозитории лежат только код сбора и помесячные сводки, а результаты разметки — в `results/news/`. Привязку к месту можно проверить на любом своём заголовке:

In [ ]:
from src.news.geo import build_gazetteer, tag
gaz = build_gazetteer(ref)
ex = pd.Series(["В Орске эвакуировали жителей из-за прорыва дамбы", "В Туве ввели режим ЧС", "Жители Волгореченска пожаловались на закрытие ГРЭС"])
mo_ids, regs = tag(ex, gaz)
pd.DataFrame({"заголовок": ex, "МО": [[ref.mo_name.get(i, i) for i in x] for x in mo_ids], "регион": regs})

**Как размечали «крупные события».** Сначала словарь ключевых слов, потом классификатор на эмбеддингах rubert-tiny2 (находит похожие по смыслу заголовки без слов из словаря), потом языковая модель Qwen3-1.7B проверяет кандидатов. Качество оценили на выборке из 200 заголовков, размеченных вручную:

In [ ]:
q = Path("data/annotation/labeling_quality.csv")
pd.read_csv(q) if q.exists() else print("оценка качества ещё не посчитана: python -m src.news.annotation score")

**Правило масштаба и длительности.** Первая версия инструкции для Qwen пропускала много мелочей — отключение горячей воды, ДТП, пожар в посёлке. Когда мы вручную проверяли выборку, договорились: крупное событие затрагивает больше ~100 человек и длится дольше нескольких дней. Перекрытая на сутки дорога или остановка транспорта на день-два на траты за месяц не влияют (при дневных данных о продажах — наоборот, могли бы помочь). Это правило мы добавили в инструкцию модели и перепроверили все её прошлые «Да». Вот сама инструкция и примеры:

In [ ]:
from src.news import llm_verify
print(llm_verify.SYSTEM, "\n")
pd.DataFrame(llm_verify.SHOTS, columns=["пример заголовка", "ответ"])

In [ ]:
llm = pd.read_parquet("data/external/news/major_llm.parquet")
old = pd.read_parquet("data/interim/major_llm_v1.parquet")   # разметка до строгого правила
print(f"Кандидатов от словаря и rubert-tiny2: {len(llm):,}")
print(f"Qwen, первая инструкция: «Да» у {int(old.llm_major.sum()):,}")
print(f"Qwen, строгое правило: «Да» у {int(llm.llm_major.sum()):,}")
pd.read_parquet("data/interim/recheck_examples.parquet")

Модель не безупречна: среди оставленных попадаются учения («условно») и новости о том, что паводок прошёл без происшествий. Поэтому точность каскада на ручной выборке — 0,82, а не 1. Для объяснения шоков этого достаточно: аналитик всё равно смотрит на заголовки сам.

**Помогают ли новости находить шоки?** Смотрим, как ведут себя траты муниципалитетов до и после крупного события — по сравнению со случайными МО:

In [ ]:
me = pd.read_pickle("data/interim/major_events.pkl")
pd.DataFrame({k: {"событий": v["n"], "после − до": round(v["delta"], 2), "у случайных МО": round(v["null"], 2), "p": round(v["p"], 3)} for k, v in me["res"].items()}).T

Для словаря и для нейросетевой разметки посчитано по-разному: по каждому МО отдельно и по событиям целиком (регион-месяц), иначе событие в Москве весит как 143 паводка. Разница «после − до» в ожидаемую сторону есть, но незначима: систематической связи «крупное событие → сдвиг месячных трат» на этих данных мы не нашли. График ниже — профиль расхождения по месяцам вокруг события:

In [ ]:
figures.event_study(); Image("report/figures/07_event_study.png")

Зато в отдельных случаях новости хорошо объясняют, что произошло. Орск: прорыв дамбы в апреле 2024 года, а в мае–июне траты заметно выше прогноза. Поэтому новости мы используем так: детектор становится чувствительнее в регионе после крупного события, а к каждой тревоге подбираются новости.

In [ ]:
figures.news_case(); Image("report/figures/06_news_case_orsk.png")

In [ ]:
ni = pd.read_pickle("data/interim/changepoint/news_informed.pkl")
al = ni["alarms"].pivot(index="territory_id", columns="режим", values="alarm")
print(f"Порог обычный {ni['thr']:.2f}, после новости о крупном событии в регионе — {ni['thr_low']:.2f}")
print("Тревог всего:", al.notna().sum().to_dict())
o = ni["events_eval"]
o.pivot_table(index="named", columns="режим", values="найден", aggfunc="mean").rename(index={True: "МО, названные в новостях", False: "все МО региона события"}).round(3)

In [ ]:
print(open("report/shock_cards.md", encoding="utf-8").read()[:3000])

## 8. Ключевая ставка
Ставка одна для всей страны, поэтому различий между муниципалитетами она не объясняет. Мы проверили её там, где она может помочь, — в прогнозе роста трат России, на ряде с 2018 года:

In [ ]:
run("src.eval.key_rate_check")
kr = pd.read_parquet("data/interim/key_rate_check.parquet")
kr.pivot_table(index="метод", columns="h", values="err").round(2).rename_axis("ошибка прогноза годового роста, п.п.")

## Сверка с презентацией
Все ключевые числа со слайдов — рядом с тем, что ноутбук только что посчитал сам. Небольшие отличия возможны только в последнем знаке из-за округления.

In [ ]:
from src.eval.metrics import summarize
slides = {  # число на слайде
    ("MAE, 1 / 3 / 6 / 12 мес.", "Prophet"): "1 805 / 1 755 / 2 159 / 2 901",
    ("MAE, 1 / 3 / 6 / 12 мес.", "Сезонная наивная с дрейфом"): "1 250 / 1 108 / 1 143 / 1 416",
    ("MAE, 1 / 3 / 6 / 12 мес.", "Chronos-2 дообученный"): "1 068 / 1 314 / 1 533 / 1 614",
    ("MAE, 1 / 3 / 6 / 12 мес.", "Ансамбль"): "1 010 / 1 131 / 1 114 / 1 416",
    ("Ансамбль точнее Prophet", "на 1 / 3 / 6 / 12 мес."): "44 / 36 / 48 / 51 %",
    ("Тест Диболда–Мариано", "ансамбль vs Prophet, 1 мес., p"): "0,02",
    ("Выбор модели (Борда и Коупленд)", "первое место"): "Ансамбль / Ансамбль",
    ("Детектор по остаткам прогноза", "находит / задержка / до шока / на выброс"): "61 % / 1,4 мес. / 2 % / 5 %",
    ("Выбор детектора (Борда и Коупленд)", "первое место"): "По остаткам прогноза / По остаткам прогноза",
    ("Разметка крупных событий", "точность каскада"): "0,82",
}
fmt = lambda xs: " / ".join(f"{x:,.0f}".replace(",", " ") for x in xs)
mae = {m: summarize(pd.read_parquet(FC / f"{m}.parquet")).set_index("h").MAE for m in ["prophet", "snaive_natg", "chronos2_ft_mix", "ens_inv_mae"]}
sel_m, sel_d = selection.models(), selection.detectors()
b = pd.read_csv("data/interim/changepoint/benchmark_summary.csv", index_col=0).loc["base_residual"]
sg = pd.read_csv("data/interim/significance.csv").query("A == 'ens_inv_mae' and B == 'prophet' and h == 1").p_DM.iloc[0]
q = pd.read_csv("data/annotation/labeling_quality.csv").iloc[-1]["точность"]
now = [fmt(mae["prophet"]), fmt(mae["snaive_natg"]), fmt(mae["chronos2_ft_mix"]), fmt(mae["ens_inv_mae"]),
       " / ".join(f"{(1 - mae['ens_inv_mae'][h] / mae['prophet'][h]) * 100:.0f}" for h in (1, 3, 6, 12)) + " %",
       f"{sg:.2f}".replace(".", ","),
       f"{sel_m['Борда (точность)'].idxmin().split(' (')[0]} / {sel_m['Коупленд (точность)'].idxmax().split(' (')[0]}",
       f"{b['доля_найденных']:.0%} / {b['задержка_мес']:.1f} мес. / {b['тревога_до_шока']:.0%} / {b['реакция_на_выброс']:.0%}".replace(".", ",").replace("%", " %"),
       f"{sel_d['Борда'].idxmin()} / {sel_d['Коупленд'].idxmax()}",
       f"{q:.2f}".replace(".", ",")]
chk = pd.DataFrame({"на слайде": list(slides.values()), "посчитано сейчас": now}, index=pd.MultiIndex.from_tuples(slides.keys()))
chk["совпало"] = np.where(chk["на слайде"].str.replace(" ", "") == chk["посчитано сейчас"].str.replace(" ", ""), "✓", "≈")
chk

## 9. Итоги
Общая динамика страны объясняет почти всё движение трат в муниципалитетах, и на ней построен прогноз, который на 36–51 % точнее Prophet. Нейросети на коротких рядах заработали после backcasting, учёта формы года и дообучения. Шоки лучше всего находит детектор по остаткам прогноза — он почти не путает их с разовыми выбросами. Новости привязаны к данным по времени и месту и помогают объяснять найденные сдвиги.

Подробнее — в [отчёте](../report/REPORT.md) и [презентации](../presentation/presentation.pdf).